In [ ]:
# 1. INSTALL
!pip install -q langchain langchain-community langchain-huggingface transformers sentence-transformers faiss-cpu pypdf langchain-text-splitters

In [ ]:
# 2. IMPORTS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from transformers import pipeline
from google.colab import files

In [ ]:
# 3. UPLOAD PDF
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

Saving DL03_Unit_3.pdf to DL03_Unit_3 (19).pdf


In [ ]:
# 4. LOAD PDF
loader = PyPDFLoader(pdf_path)
documents = loader.load()

In [ ]:
# 5. SPLIT TEXT
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
docs = splitter.split_documents(documents)

In [ ]:
# 6. EMBEDDINGS (OFFLINE)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# 7. VECTOR DB
db = FAISS.from_documents(docs, embeddings)
retriever = db.as_retriever(search_kwargs={"k": 3})

In [ ]:
# 8. LLM (FIXED VERSION)
pipe = pipeline(
    "text-generation",   # ✅ FIXED (IMPORTANT)
    model="google/flan-t5-base",
    max_new_tokens=256,
    do_sample=True
)

llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

In [ ]:
# 9. PROMPT
prompt = PromptTemplate.from_template(
"""You are a helpful assistant.

Use ONLY the context below.

Context:
{context}

Question:
{question}

Answer:"""
)


In [ ]:
# 10. RAG FUNCTION
def ask_question(query):
    docs = retriever.invoke(query)
    context = " ".join([d.page_content for d in docs])

    final_prompt = prompt.invoke({
        "context": context,
        "question": query
    })

    return llm.invoke(final_prompt)

In [ ]:
# 11. CHAT LOOP
print("\n📚 RAG Chatbot Ready (type exit to stop)\n")

while True:
    q = input("Ask question: ")
    if q.lower() == "exit":
        break

    print("\nAnswer:\n", ask_question(q), "\n")


📚 RAG Chatbot Ready (type exit to stop)

Ask question: What is GRU


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 You are a helpful assistant.

Use ONLY the context below.

Context:
GRU Architecture GRU
● Alternative to LSTM, computationally less expensive and faster to train
● Uses gating mechanisms to selectively update the hidden state of the 
network at each time step
● The gating mechanisms are used to control the ﬂow of information in and out 
of the network. 
The GRU has two gating mechanisms,
 i) Reset Gate  
ii) Update Gate Gated Recurrent Unit
(GRU)
Alternative for LSTM

Question:
What is GRU

Answer:ower generations for data network interface routing in xanvas virtualization and network control implementation environments around devices in realmation technology design ecosystem including microservice infrastructure technology as hardware components in communications operations cluster processor architecture (AVRSTC and LCLSDQ2) architecture systems from LGIT in virtual world systems systems hardware component management design. systems architecture. construct-or select device 

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 You are a helpful assistant.

Use ONLY the context below.

Context:
Basics of Sequential Model
- Specially designed for sequential data
- Input is Sequential Data (sequence matters)
e.g. Textual statement (Natural Language)
        Time-Series
        Speech
        DNA Sequence Sequential Data Handling
Standard Feedforward Model → Each input data is treated as independent element
Sequential Model → Order of elements is as important as data itself, 
          → Maintain a "memory" of what has occurred previously 
                                   to inform the current output.
Sequential data → data such as text, audio, or time-series sensor data, a single data 
point (e.g., a word) lacks full meaning without its context (the preceding words) Sequential Models
Unit 03, 01AML307: Deep Learning

Question:
What is Sequential models

Answer: 

Ask question: exit
